# TASK 2 · Customer Segmentation Analysis

**Objective:** Segment an e-commerce customer base using purchasing behavior for targeted marketing.

**Tech:** Python, pandas, NumPy, scikit-learn KMeans, matplotlib, seaborn, Jupyter Notebook.

This notebook uses transaction-level data to build **RFM (Recency, Frequency, Monetary)** customer features, standardize them, determine a suitable K with the Elbow Method, and profile the resulting customer segments.

In [ ]:
# 1. Import libraries
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

print("Libraries imported successfully.")

In [ ]:
# 2. Load the dataset
# Put the Online Retail CSV in the same folder as this notebook.
DATA_PATH = "data.csv"

df = pd.read_csv(DATA_PATH, encoding="latin1")

print("Dataset shape:", df.shape)
display(df.head())

## 3. Inspect the dataset

Check columns, data types, missing values, duplicates, and descriptive statistics before cleaning.

In [ ]:
print("Columns:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes.to_frame("dtype"))

print("\nMissing values:")
display(df.isnull().sum().to_frame("missing_values"))

print("\nDuplicate rows:", df.duplicated().sum())

display(df.describe(include="all").T)

## 4. Data cleaning

For RFM analysis, customers need valid IDs and transactions need valid dates, positive quantities, and positive prices. Returns/cancellations are excluded so the clusters represent purchasing behavior.

In [ ]:
df_clean = df.copy()
df_clean.columns = [str(c).strip() for c in df_clean.columns]

df_clean = df_clean.drop_duplicates()
df_clean["InvoiceDate"] = pd.to_datetime(df_clean["InvoiceDate"], errors="coerce")

df_clean = df_clean.dropna(
    subset=["CustomerID", "InvoiceDate", "Quantity", "UnitPrice"]
)

df_clean = df_clean[
    (df_clean["Quantity"] > 0) &
    (df_clean["UnitPrice"] > 0)
].copy()

df_clean["TotalAmount"] = df_clean["Quantity"] * df_clean["UnitPrice"]

print("Cleaned shape:", df_clean.shape)
display(df_clean.head())

**Observation:** The cleaned dataset contains usable customer and transaction information, making it suitable for customer-level behavioral analysis.

In [ ]:
# 5. Descriptive customer statistics

customer_summary = df_clean.groupby("CustomerID").agg(
    Average_Purchase_Value=("TotalAmount", "mean"),
    Purchase_Frequency=("InvoiceNo", "nunique"),
    Customer_Lifetime_Value=("TotalAmount", "sum")
).reset_index()

print("Unique customers:", customer_summary["CustomerID"].nunique())
display(customer_summary.describe().T)

display(customer_summary[
    ["Average_Purchase_Value", "Purchase_Frequency", "Customer_Lifetime_Value"]
].mean().to_frame("Mean"))

## 6. RFM feature engineering

- **Recency:** days since the customer's latest purchase.
- **Frequency:** number of unique invoices/orders.
- **Monetary:** total historical spending.

The lifetime value field below is a **historical CLV proxy**, not a prediction of future revenue.

In [ ]:
snapshot_date = df_clean["InvoiceDate"].max() + pd.Timedelta(days=1)

rfm = df_clean.groupby("CustomerID").agg(
    Recency=("InvoiceDate", lambda x: (snapshot_date - x.max()).days),
    Frequency=("InvoiceNo", "nunique"),
    Monetary=("TotalAmount", "sum")
).reset_index()

rfm["AveragePurchaseValue"] = rfm["Monetary"] / rfm["Frequency"]
rfm["CustomerLifetimeValueProxy"] = rfm["Monetary"]

display(rfm.head())
display(rfm[[
    "Recency", "Frequency", "Monetary",
    "AveragePurchaseValue", "CustomerLifetimeValueProxy"
]].describe().T)

**Observation:** RFM summarizes each customer's purchasing behavior. Lower Recency means more recent activity, while higher Frequency and Monetary indicate stronger engagement and spending.

In [ ]:
# 7. Select behavioral features
features = ["Recency", "Frequency", "Monetary"]
X = rfm[features].copy()

# Log transformation reduces the effect of extreme skew.
X_log = np.log1p(X)

print("Selected features:", features)
display(X.describe().T)

In [ ]:
# 8. Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_log)

X_scaled_df = pd.DataFrame(X_scaled, columns=features)
display(X_scaled_df.describe().T)

## 9. Elbow Method

K-Means inertia is calculated for several K values. Look for the point where additional clusters provide diminishing improvement.

In [ ]:
inertias = []
k_values = range(2, 11)

for k in k_values:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    model.fit(X_scaled)
    inertias.append(model.inertia_)

plt.figure(figsize=(9, 5))
plt.plot(k_values, inertias, marker="o")
plt.title("Elbow Method for Optimal K")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia")
plt.xticks(list(k_values))
plt.show()

**Observation:** The elbow indicates a practical point at which increasing K gives smaller improvements in within-cluster compactness. Use the visible elbow as the primary guide.

In [ ]:
# 10. Supporting silhouette-score check

scores = []
for k in range(2, 11):
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(X_scaled)
    scores.append((k, silhouette_score(X_scaled, labels)))

silhouette_df = pd.DataFrame(scores, columns=["K", "Silhouette_Score"])
display(silhouette_df)

plt.figure(figsize=(9, 5))
plt.plot(silhouette_df["K"], silhouette_df["Silhouette_Score"], marker="o")
plt.title("Silhouette Score by K")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Silhouette Score")
plt.xticks(range(2, 11))
plt.show()

print("Highest silhouette score K:",
      int(silhouette_df.loc[silhouette_df["Silhouette_Score"].idxmax(), "K"]))

In [ ]:
# 11. Final K-Means model
# Review the Elbow plot first. Change this if another K is clearly better.
optimal_k = 4

kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
rfm["Cluster"] = kmeans.fit_predict(X_scaled)

print("K-Means completed with K =", optimal_k)
display(rfm["Cluster"].value_counts().sort_index().to_frame("Customers"))

## 12. Cluster visualization — Frequency vs Monetary

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=rfm,
    x="Frequency",
    y="Monetary",
    hue="Cluster",
    palette="tab10",
    s=70,
    alpha=0.75
)
plt.title("Customer Segments: Frequency vs Monetary")
plt.xlabel("Purchase Frequency")
plt.ylabel("Monetary Value")
plt.legend(title="Cluster")
plt.show()

**Observation:** Customers toward the upper-right have both high purchase frequency and high historical spending, making them strong candidates for retention and loyalty programs.

## 13. Cluster visualization — Recency vs Monetary

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=rfm,
    x="Recency",
    y="Monetary",
    hue="Cluster",
    palette="tab10",
    s=70,
    alpha=0.75
)
plt.title("Customer Segments: Recency vs Monetary")
plt.xlabel("Recency (Days Since Last Purchase)")
plt.ylabel("Monetary Value")
plt.legend(title="Cluster")
plt.show()

**Observation:** Lower Recency represents more recent activity. Customers with high Monetary value but high Recency may be valuable customers who are becoming inactive and should receive win-back campaigns.

## 14. Cluster profiling

In [ ]:
cluster_profile = rfm.groupby("Cluster").agg(
    Customers=("CustomerID", "count"),
    Avg_Recency=("Recency", "mean"),
    Avg_Frequency=("Frequency", "mean"),
    Avg_Monetary=("Monetary", "mean"),
    Avg_Purchase_Value=("AveragePurchaseValue", "mean")
).round(2)

display(cluster_profile)

In [ ]:
# 15. Cluster profile heatmap
profile = cluster_profile[[
    "Avg_Recency", "Avg_Frequency", "Avg_Monetary"
]]

profile_z = pd.DataFrame(
    StandardScaler().fit_transform(profile),
    index=profile.index,
    columns=profile.columns
)

plt.figure(figsize=(9, 5))
sns.heatmap(profile_z, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Standardized Cluster Profile")
plt.xlabel("RFM Metric")
plt.ylabel("Cluster")
plt.show()

**Observation:** The profile heatmap makes it easier to compare clusters relative to one another. High Frequency/Monetary with low Recency generally indicates highly engaged customers.

## 16. Customers per cluster

In [ ]:
cluster_counts = rfm["Cluster"].value_counts().sort_index()

plt.figure(figsize=(8, 5))
sns.barplot(x=cluster_counts.index.astype(str), y=cluster_counts.values)
plt.title("Number of Customers per Cluster")
plt.xlabel("Cluster")
plt.ylabel("Number of Customers")
plt.show()

display(cluster_counts.to_frame("Customers"))

**Observation:** Cluster size shows how the customer base is distributed. Segment size should be considered together with customer value when allocating marketing resources.

## 17. Customer type interpretation and marketing actions

K-Means cluster numbers are arbitrary, so interpret each cluster using the profile table rather than assuming Cluster 0, 1, 2, or 3 has a fixed meaning.

| Customer Type | Typical RFM Pattern | Marketing Action |
|---|---|---|
| **Champions / High Value** | Low Recency + high Frequency + high Monetary | VIP rewards, loyalty benefits, early access, premium recommendations |
| **Loyal Customers** | Relatively recent + strong Frequency | Cross-sell, bundles, loyalty points, personalized offers |
| **Potential Loyalists** | Recent + moderate Frequency/Monetary | Repeat-purchase incentives and recommendations |
| **At-Risk High Value** | High Recency + historically high Frequency/Monetary | Personalized win-back offers and reminders |
| **Hibernating / Low Value** | High Recency + low Frequency/Monetary | Low-cost reactivation campaigns |

Match these descriptions to the actual cluster averages shown in the profiling table.

## 18. Key insights

1. RFM transforms transaction history into actionable customer segments.
2. Recency identifies current engagement and possible inactivity.
3. Frequency highlights repeat-purchase behavior.
4. Monetary identifies high-value customers.
5. Standardization prevents a large-scale feature from dominating K-Means.
6. Segment size and customer value should both guide marketing investment.
7. RFM segments should be recalculated periodically as new transactions arrive.

# Conclusion

The analysis uses **RFM features and K-Means clustering** to divide an e-commerce customer base into behaviorally similar groups. These segments can support targeted retention, loyalty, cross-selling, and win-back strategies.

### Final recommendations
- Reward high-value and loyal customers.
- Encourage promising customers to purchase again.
- Prioritize win-back campaigns for valuable but inactive customers.
- Use inexpensive reactivation campaigns for low-value inactive customers.
- Refresh the segmentation regularly to reflect new purchasing behavior.

**Submission note:** `optimal_k = 4` is a starting value. Review the Elbow plot and change it if the dataset's elbow clearly suggests another K.